# Milestone 1

This milestone focuses on understanding the dataset and establishing a baseline performance through **exploratory data analysis (EDA)** and simple **heuristic-based methods** using `librosa`.

---

## Suggested Readings
- [Hugging Face Audio Course](https://huggingface.co/learn/audio-course/en/chapter0/introduction)
- [Librosa Documentation](https://librosa.org/doc/main/core.html#audio-loading)

---

## Instructions
Use this notebook to answer **all Milestone-1 questions**.

---

## Resources
- Notebook Link:  
  https://colab.research.google.com/drive/1m6UczhxQIke_raWSqukSWuiKbIVt7MMb?usp=sharing  

- Competition Link:  
  https://www.kaggle.com/competitions/jan-2026-dl-gen-ai-project/


In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import torch

import warnings
warnings.filterwarnings("ignore")

In [ ]:
#----------------------------- DON'T CHANGE THIS --------------------------
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB = 20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

In [ ]:
# CONFIGURATION
DATA_ROOT = '/kaggle/input/jan-2026-dl-gen-ai-project/'

# All genres in alphabetical order
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']

# Stems file names (dict keys are filenames, iterated to produce stem names)
STEMS = {
    'bass.wav': None,
    'drums.wav': None,
    'other.wav': None,
    'vocals.wav': None
}

STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
GENRE_TO_TEST = 'rock'
SONG_INDEX = 0   # First song from rock genre (Q10)

## `build_dataset` — Q1 to Q3

Builds train/val dictionaries from the dataset root.

**Checks performed:**
1. **Completeness** — all 4 stem `.wav` files must be present in a song folder.
2. **Corruption** — any stem file smaller than **4 KB (4096 bytes)** marks the whole song as corrupted.
3. **Stratified shuffle split** — `val_split=0.17` per genre using `DATA_SEED`.

In [ ]:
def build_dataset(root_dir, val_split=0.17, seed=DATA_SEED):
    """
    Walk through root_dir, validate songs, and split into train/val.

    Expected structure:
        root_dir/
          <genre>/
            <song_folder>/
              bass.wav
              drums.wav
              other.wav
              vocals.wav

    Returns
    -------
    train_dataset : dict  {genre: {stem_name: [file_paths]}}
    val_dataset   : dict  {genre: {stem_name: [file_paths]}}
    """

    # Initialise empty dicts — stem key is filename without extension
    train_dataset = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    val_dataset   = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}

    rng = random.Random(seed)

    corrupted_songs  = 0
    incomplete_songs = 0
    total_valid      = 0

    for genre in GENRES:
        genre_dir = os.path.join(root_dir, genre)

        # CHECK: genre folder exists
        if not os.path.isdir(genre_dir):
            print(f"[WARNING] Genre folder not found: {genre_dir}")
            continue

        # Collect all song sub-folders for this genre
        song_folders = sorted([
            d for d in os.listdir(genre_dir)
            if os.path.isdir(os.path.join(genre_dir, d))
        ])

        valid_songs = []   # list of song folder paths that pass all checks

        for song in song_folders:
            song_path = os.path.join(genre_dir, song)

            # ---- CHECK: Completeness — all stems must exist ----
            stem_files = {s: os.path.join(song_path, s) for s in STEMS}
            if not all(os.path.isfile(p) for p in stem_files.values()):
                incomplete_songs += 1
                continue

            # ---- CHECK: Corruption — any file < 4 KB (4096 bytes) ----
            # Hint: 1kb = 1024 bytes  =>  4kb = 4096 bytes
            is_corrupt = any(
                os.path.getsize(p) < 4 * 1024   # 4 KB threshold
                for p in stem_files.values()
            )
            if is_corrupt:
                corrupted_songs += 1
                continue

            valid_songs.append(song_path)

        total_valid += len(valid_songs)

        # ---- Stratified Shuffle Split ----
        rng.shuffle(valid_songs)
        n_val = max(1, int(round(len(valid_songs) * val_split)))
        val_songs   = valid_songs[:n_val]
        train_songs = valid_songs[n_val:]

        # ---- Helper: populate a split dict ----
        def add_to_dict(target_dict, song_list):
            for sp in song_list:
                for stem_filename in STEMS:
                    stem_key  = stem_filename.replace('.wav', '')
                    file_path = os.path.join(sp, stem_filename)
                    target_dict[genre][stem_key].append(file_path)

        add_to_dict(train_dataset, train_songs)
        add_to_dict(val_dataset,   val_songs)

    print(f"Total valid songs   : {total_valid}")
    print(f"Corrupted songs     : {corrupted_songs}")
    print(f"Incomplete songs    : {incomplete_songs}")

    return train_dataset, val_dataset


tr, val = build_dataset(DATA_ROOT)

### Q1 — Size Distribution Analysis
Count songs in different file-size buckets and answer:
> `corrupted_songs + songs_less_than_5.0491MB`

In [ ]:
# ---- File-size analysis across the entire dataset ----
# We inspect mixture / stem files to find size distribution.
# Here we look at the mixture (mix) file size per song folder.

THRESHOLD_LOW_MB  = 5.0491   # MB
THRESHOLD_HIGH_MB = 5.0493   # MB

file_sizes_mb = []

for genre in GENRES:
    genre_dir = os.path.join(DATA_ROOT, genre)
    if not os.path.isdir(genre_dir):
        continue
    for song in sorted(os.listdir(genre_dir)):
        song_path = os.path.join(genre_dir, song)
        if not os.path.isdir(song_path):
            continue
        for stem_fn in STEMS:
            fp = os.path.join(song_path, stem_fn)
            if os.path.isfile(fp):
                size_mb = os.path.getsize(fp) / (1024 * 1024)
                file_sizes_mb.append(size_mb)

file_sizes_mb = np.array(file_sizes_mb)

songs_less_than_low  = int(np.sum(file_sizes_mb < THRESHOLD_LOW_MB))
songs_greater_than_high = int(np.sum(file_sizes_mb > THRESHOLD_HIGH_MB))

print(f"Songs < {THRESHOLD_LOW_MB} MB  : {songs_less_than_low}")
print(f"Songs > {THRESHOLD_HIGH_MB} MB  : {songs_greater_than_high}")
print()

# Count corrupted songs (< 4 KB) separately
corrupted_count = int(np.sum(np.array([os.path.getsize(os.path.join(DATA_ROOT, g, s, sf))
    for g in GENRES
    for s in (os.listdir(os.path.join(DATA_ROOT, g)) if os.path.isdir(os.path.join(DATA_ROOT, g)) else [])
    for sf in STEMS
    if os.path.isfile(os.path.join(DATA_ROOT, g, s, sf))
]) < 4 * 1024))

print(f"Corrupted files (< 4 KB) : {corrupted_count}")
print()
print(f"Q1 Answer (corrupted + songs < {THRESHOLD_LOW_MB} MB) = {corrupted_count + songs_less_than_low}")
print(f"Q2 Answer |songs > {THRESHOLD_HIGH_MB}MB  -  songs < {THRESHOLD_LOW_MB}MB| = {abs(songs_greater_than_high - songs_less_than_low)}")

### Q3 — Training reggae drums vs. Validation country vocals

In [ ]:
n_train_reggae_drums   = len(tr['reggae']['drums'])
n_val_country_vocals   = len(val['country']['vocals'])

print(f"Training reggae drum samples   : {n_train_reggae_drums}")
print(f"Validation country vocal samples: {n_val_country_vocals}")
print(f"Absolute difference (Q3)        : {abs(n_train_reggae_drums - n_val_country_vocals)}")

---
## `find_long_silences` — Q4 to Q9

In [ ]:
def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):
    """
    Scan every audio file in dataset_dict and record those whose
    longest continuous silence segment is >= threshold_sec seconds.

    Parameters
    ----------
    dataset_dict  : dict  {genre: {stem_name: [file_paths]}}
    sr            : int   target sample rate
    threshold_sec : float minimum silence length (seconds) to flag
    top_db        : float threshold (dB) below reference to classify as silence
                         (passed directly to librosa.effects.split)

    Returns
    -------
    df : pd.DataFrame with columns
         Genre | Stem | Duration | Max_Silence_Sec | Silence_Location | File_Path
    """

    records = []

    # ---- COUNT TOTAL FILES ----
    total_files = sum(
        len(paths)
        for genre_dict in dataset_dict.values()
        for paths in genre_dict.values()
    )
    print(f"Total files to scan: {total_files}")

    with tqdm(total=total_files, desc="Scanning for silence") as pbar:
        for genre, stems in dataset_dict.items():
            for stem_name, file_paths in stems.items():
                for file_path in file_paths:
                    pbar.update(1)
                    try:
                        # ---- Load Audio ----
                        y, _ = librosa.load(file_path, sr=sr, mono=True)
                        total_duration = len(y) / sr   # total seconds

                        # ---- Find Non-Silent Intervals ----
                        # Returns array of shape (n_intervals, 2) — [start, end] sample indices
                        non_silent = librosa.effects.split(y, top_db=top_db)

                        max_silence   = 0.0
                        silence_type  = []

                        # ---- CASE A: Fully silent (no non-silent intervals found) ----
                        if len(non_silent) == 0:
                            max_silence = total_duration
                            silence_type.append('full')

                        else:
                            # ---- CASE B: START silence ----
                            start_silence_sec = non_silent[0][0] / sr
                            if start_silence_sec > 0:
                                if start_silence_sec > max_silence:
                                    max_silence = start_silence_sec
                                silence_type.append('start')

                            # ---- CASE C: END silence ----
                            end_silence_sec = (len(y) - non_silent[-1][1]) / sr
                            if end_silence_sec > 0:
                                if end_silence_sec > max_silence:
                                    max_silence = end_silence_sec
                                silence_type.append('end')

                            # ---- CASE D: MIDDLE silence (gaps between non-silent intervals) ----
                            for i in range(len(non_silent) - 1):
                                gap_sec = (non_silent[i + 1][0] - non_silent[i][1]) / sr
                                if gap_sec > 0:
                                    if gap_sec > max_silence:
                                        max_silence = gap_sec
                                    if 'middle' not in silence_type:
                                        silence_type.append('middle')

                        # ---- Store result if silence meets threshold ----
                        if max_silence >= threshold_sec:
                            records.append({
                                "Genre": genre,
                                "Stem": stem_name,
                                "Duration": round(total_duration, 2),
                                "Max_Silence_Sec": round(max_silence, 2),
                                "Silence_Location": ", ".join(silence_type),
                                "File_Path": file_path
                            })

                    except Exception as e:
                        print(f"[ERROR] {file_path}: {e}")

    df = pd.DataFrame(records)
    return df


# --- EXECUTION ---
df_silence = find_long_silences(tr, threshold_sec=DURATION, top_db=TOP_DB)
print(f"\nTotal files with silence >= {DURATION}s : {len(df_silence)}")

In [ ]:
# --- RESULTS ANALYSIS ---

# Q4: Total files with silence >= 5s
print("=" * 60)
print(f"Q4  Total files with silence >= 5s  : {len(df_silence)}")

# Q5: Total vocals with silence >= 5s
vocals_silence = df_silence[df_silence['Stem'] == 'vocals']
print(f"Q5  Total Vocals with silence >= 5s : {len(vocals_silence)}")

# Q6: Average silence length in Vocals (seconds)
avg_vocal_silence = vocals_silence['Max_Silence_Sec'].mean()
print(f"Q6  Avg Silence Length in Vocals    : {avg_vocal_silence:.2f} s")

# Q7: Jazz drums with silence >= 5s
jazz_drums_silence = df_silence[(df_silence['Genre'] == 'jazz') & (df_silence['Stem'] == 'drums')]
print(f"Q7  Jazz drums with silence >= 5s   : {len(jazz_drums_silence)}")

# Q8: Jazz drums with silence >= 5s AND Silence_Location is ONLY 'middle'
jazz_drums_middle = jazz_drums_silence[
    jazz_drums_silence['Silence_Location'].str.strip() == 'middle'
]
print(f"Q8  Jazz drums silence only middle  : {len(jazz_drums_middle)}")

# Q9: Jazz drums with silence >= 5s AND Max_Silence_Sec >= 10
jazz_drums_10s = jazz_drums_silence[jazz_drums_silence['Max_Silence_Sec'] >= 10]
print(f"Q9  Jazz drums Max_Silence >= 10s   : {len(jazz_drums_10s)}")
print("=" * 60)

In [ ]:
# Pivot Table: Count of silence files by Genre vs Stem
if not df_silence.empty:
    pivot = df_silence.pivot_table(
        index='Genre',
        columns='Stem',
        values='File_Path',
        aggfunc='count',
        fill_value=0
    )
    print("\nPivot Table — Silence count by Genre vs Stem:")
    print(pivot)
else:
    print("No silence records found.")

---
## Q10 – Q12 : Rock song stem mixing

- Load the **first song** (`SONG_INDEX = 0`) from the `rock` genre.
- Stack all 4 stems, mix by element-wise sum.
- Compute **RMS amplitude** and apply **peak normalisation**.

In [ ]:
stems_audio = []
try:
    for key in STEM_KEYS:
        # Retrieve the file path for this stem from the training dict
        file_path = tr[GENRE_TO_TEST][key][SONG_INDEX]

        # Load audio: fixed Duration=5.0s, SR=22050 for speed & consistency
        y, _ = librosa.load(file_path, sr=SR, duration=DURATION, mono=True)
        stems_audio.append(y)
        print(f"Loaded [{key}]: {os.path.basename(file_path)} — {len(y)} samples")

    print("\nAudio loaded successfully.")

except NameError:
    print("ERROR: 'tr' dictionary not found. Please run build_dataset() first.")
except IndexError:
    print(f"ERROR: Song index {SONG_INDEX} out of range for genre '{GENRE_TO_TEST}'.")
except Exception as e:
    print(f"ERROR: {e}")

In [ ]:
# ---- Stack stems into a numpy array (Shape: 4 x Samples) ----
stems_stack = np.array(stems_audio)          # shape: (4, n_samples)
print(f"stems_stack shape : {stems_stack.shape}")

# ---- Mix stems by summing element-wise along axis-0 ----
mix_raw = np.sum(stems_stack, axis=0)        # shape: (n_samples,)
print(f"mix_raw length    : {len(mix_raw)}")

# ---- Calculate RMS Amplitude MANUALLY ----
# RMS = sqrt( mean( x^2 ) )
rms_val = np.sqrt(np.mean(mix_raw ** 2))
print(f"RMS amplitude     : {rms_val:.4f}   → rounded: {round(rms_val, 2)}")

# ---- Peak Normalization ----
# Divide by the maximum absolute value so the peak becomes ±1.0
max_val = np.max(np.abs(mix_raw))
print(f"Max absolute value: {max_val:.6f}")

if max_val > 0:
    mix_norm = mix_raw / max_val
else:
    mix_norm = mix_raw

# VALIDATION: absolute peak must equal 1.0
assert np.isclose(np.max(np.abs(mix_norm)), 1.0), "Normalization failed."
print("Peak normalisation assertion passed ✓")

# Q12: max value (not max absolute value)
max_norm = np.max(mix_norm)
print(f"\nQ10  Length of mix sample             : {len(mix_raw)}")
print(f"Q11  RMS amplitude of mix             : {round(rms_val, 2)}")
print(f"Q12  Max value of peak-norm'd sample  : {round(max_norm, 2)}")